# Matrix Operations - Theory & Implementations

This notebook follows the `Matrix Operations` chapter, but in the order that is easiest to learn by doing:

1. Matrix shapes, notation, and special matrices
2. Elementwise operations, trace, and transpose
3. Matrix multiplication from row, column, and outer-product views
4. Dense layers and attention as matrix operations
5. Inverse, condition number, and solving linear systems
6. Pseudo-inverse, least squares, and projection
7. LU, QR, and eigendecomposition
8. SVD, low-rank approximation, and LoRA intuition
9. Cholesky and symmetric positive definite matrices
10. GEMM vs GEMV and the hardware view of performance

The notebook is intentionally code-first. The goal is not only to know the formulas, but to make the operations feel operational.

Primary references behind the chapter: Axler, MIT 18.06, LAPACK, cuBLAS, Vaswani et al. (2017), Dao et al. (2022, 2024), Hu et al. (2022), and Halko et al. (2011).

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)
np.random.seed(7)

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False
    print("matplotlib not available - plots will be skipped")


def softmax_rows(S):
    S = np.asarray(S, dtype=float)
    S_shifted = S - S.max(axis=1, keepdims=True)
    exp_S = np.exp(S_shifted)
    return exp_S / exp_S.sum(axis=1, keepdims=True)


def frobenius_inner(A, B):
    A = np.asarray(A, dtype=float)
    B = np.asarray(B, dtype=float)
    return float(np.trace(A.T @ B))


def outer_sum_product(A, B):
    A = np.asarray(A, dtype=float)
    B = np.asarray(B, dtype=float)
    m, p = A.shape
    p2, n = B.shape
    if p != p2:
        raise ValueError("inner dimensions must match")
    C = np.zeros((m, n), dtype=float)
    for k in range(p):
        C += np.outer(A[:, k], B[k, :])
    return C


def lu_no_pivot(A):
    A = np.array(A, dtype=float)
    n = A.shape[0]
    L = np.eye(n)
    U = A.copy()
    for k in range(n - 1):
        pivot = U[k, k]
        if abs(pivot) < 1e-12:
            raise ValueError("zero pivot encountered; pivoting required")
        for i in range(k + 1, n):
            factor = U[i, k] / pivot
            L[i, k] = factor
            U[i, k:] -= factor * U[k, k:]
    return L, U


def power_iteration(A, steps=20):
    A = np.asarray(A, dtype=float)
    v = np.random.randn(A.shape[1])
    v /= np.linalg.norm(v)
    for _ in range(steps):
        v = A @ v
        v /= np.linalg.norm(v)
    eigenvalue = float(v @ (A @ v))
    return eigenvalue, v


def truncated_svd(A, rank):
    U, S, Vt = np.linalg.svd(A, full_matrices=False)
    return U[:, :rank] @ np.diag(S[:rank]) @ Vt[:rank, :], S


---
## 1. Matrix Basics, Shapes, and Special Structure

Start concrete. A matrix is both a table of numbers and a linear map with a shape. Shape reasoning is the first debugging tool in linear algebra and in deep learning code.

Learning move: inspect rows, columns, shapes, and special matrices before touching matrix multiplication.

In [ ]:
# === 1.1 Basic matrix objects and shape reasoning ===
A = np.array([[1., 2., 3.],
              [4., 5., 6.]])
I3 = np.eye(3)
D = np.diag([2., -1., 4.])
U = np.array([[1., 2., 3.],
              [0., 4., 5.],
              [0., 0., 6.]])
S = np.array([[2., 1., 3.],
              [1., 4., 5.],
              [3., 5., 6.]])

print("A shape:", A.shape)
print("A =\n", A)
print("row 0 =", A[0, :])
print("column 1 =", A[:, 1])
print("I3 =\n", I3)
print("Diagonal matrix D =\n", D)
print("Upper triangular U =\n", U)
print("Symmetric matrix S =\n", S)
print("S.T == S ?", np.allclose(S.T, S))

# Memory layout intuition: transpose changes interpretation, not always data.
print("A strides =", A.strides)
print("A.T strides =", A.T.strides)
print("A.T shares memory with A ?", np.shares_memory(A, A.T))

---
## 2. Elementwise Operations, Trace, and Transpose

Before the main algebraic operation, matrix multiplication, there is a simpler layer of operations: entrywise arithmetic, scalar scaling, trace, and transpose.

These operations appear constantly in optimizers, masks, residual updates, and gradient formulas.

In [ ]:
# === 2.1 Elementwise operations and transpose identities ===
A = np.array([[1., 2.],
              [3., 4.]])
B = np.array([[5., 6.],
              [7., 8.]])
alpha = -0.5

print("A + B =\n", A + B)
print("A - B =\n", A - B)
print("alpha * A =\n", alpha * A)
print("Hadamard A * B =\n", A * B)
print("trace(A) =", np.trace(A))
print("trace(A.T @ B) =", np.trace(A.T @ B))
print("Frobenius inner(A, B) =", frobenius_inner(A, B))

AB = A @ B
lhs = (AB).T
rhs = B.T @ A.T
print("(AB)^T =\n", lhs)
print("B^T A^T =\n", rhs)
print("(AB)^T == B^T A^T ?", np.allclose(lhs, rhs))

---
## 3. Matrix Multiplication from Multiple Views

Matrix multiplication is the computational heart of the chapter. A single formula is not enough intuition. You want to be able to see it in at least three ways:

1. row of `A` dot column of `B`
2. `A` transforming each column of `B`
3. sum of outer products

When those three views line up in your head, many AI formulas become much easier to read.

In [ ]:
# === 3.1 Row, column, and outer-product views of AB ===
A = np.array([[1., 2., 3.],
              [4., 5., 6.]])
B = np.array([[1., 0.],
              [0., 1.],
              [1., 1.]])

C = A @ B
print("A shape:", A.shape, "B shape:", B.shape, "C shape:", C.shape)
print("C = A @ B =\n", C)

# Row-column dot product view
c00_manual = np.dot(A[0, :], B[:, 0])
c01_manual = np.dot(A[0, :], B[:, 1])
print("manual first row entries via dot products:", c00_manual, c01_manual)

# Column view: each output column is A applied to one column of B
print("A @ B[:, 0] =", A @ B[:, 0])
print("A @ B[:, 1] =", A @ B[:, 1])

# Outer-product view
C_outer = outer_sum_product(A, B)
print("sum of outer products =\n", C_outer)
print("outer-product view matches matmul ?", np.allclose(C, C_outer))

# Rank bottleneck intuition
print("rank(A) =", np.linalg.matrix_rank(A))
print("rank(B) =", np.linalg.matrix_rank(B))
print("rank(C) =", np.linalg.matrix_rank(C))

---
## 4. Dense Layers and Attention as Matrix Operations

The cleanest way to see why matrices dominate AI is to implement the core computations directly:

- batched dense layer: `Y = X W^T + b`
- single-head attention: `S = Q K^T / sqrt(d_k)`, `A = softmax(S)`, `O = A V`

Once you can read the shapes here automatically, transformer code stops feeling mysterious.

In [ ]:
# === 4.1 Dense layer and attention as matrix pipelines ===
X = np.array([[1.0, 0.5, -1.0],
              [0.0, 2.0, 1.0]])          # batch of 2, feature dim 3
W = np.array([[1.0, -1.0, 0.5],
              [0.0, 2.0, 1.0]])         # 2 output units, 3 input units
b = np.array([0.1, -0.2])

Y = X @ W.T + b
print("Dense layer shapes: X", X.shape, "W", W.shape, "b", b.shape, "Y", Y.shape)
print("Y =\n", Y)

Q = np.array([[1.0, 0.0],
              [0.5, 1.0],
              [1.0, 1.0],
              [0.0, 1.0]])
K = np.array([[1.0, 0.0],
              [1.0, 1.0],
              [0.0, 1.0],
              [0.5, 0.5]])
V = np.array([[2.0, 0.0],
              [1.0, 1.0],
              [0.0, 2.0],
              [1.5, 0.5]])

dk = Q.shape[1]
S = (Q @ K.T) / np.sqrt(dk)
A_attn = softmax_rows(S)
O = A_attn @ V

print("\nAttention score matrix S shape:", S.shape)
print(S)
print("\nAttention weights shape:", A_attn.shape)
print(A_attn)
print("row sums:", A_attn.sum(axis=1))
print("\nAttention output O shape:", O.shape)
print(O)

print("\nIf Q = K, score matrix is symmetric?", np.allclose(Q @ Q.T, (Q @ Q.T).T))

---
## 5. Inverse, Solving, and Condition Number

Mathematically, `A^{-1}` is clean. Numerically, solving `Ax = b` directly is usually better than explicitly forming the inverse.

Learning move: compare a well-conditioned matrix with an ill-conditioned one and watch how the condition number changes the problem.

In [ ]:
# === 5.1 Inverse, solve, and conditioning ===
A_good = np.array([[3.0, 1.0],
                   [1.0, 2.0]])
b = np.array([1.0, 0.0])

x_solve = np.linalg.solve(A_good, b)
x_inv = np.linalg.inv(A_good) @ b
print("A_good =\n", A_good)
print("solve(A_good, b) =", x_solve)
print("inv(A_good) @ b =", x_inv)
print("condition number of A_good =", np.linalg.cond(A_good))

# Ill-conditioned Hilbert-type example
A_bad = np.array([[1.0, 1.0],
                  [1.0, 1.0001]])
b_bad = np.array([2.0, 2.0001])
x_bad = np.linalg.solve(A_bad, b_bad)

print("\nA_bad =\n", A_bad)
print("condition number of A_bad =", np.linalg.cond(A_bad))
print("solution to A_bad x = b_bad =", x_bad)

# Tiny perturbation in b
b_bad_perturbed = b_bad + np.array([0.0, 1e-4])
x_bad_perturbed = np.linalg.solve(A_bad, b_bad_perturbed)
print("perturbed solution =", x_bad_perturbed)
print("solution change =", x_bad_perturbed - x_bad)

---
## 6. Pseudo-Inverse, Least Squares, and Projection

The pseudo-inverse is what you use when ordinary inverse is unavailable or not the right object. In practice, this means least squares, overdetermined systems, and projection onto the column space.

The geometric picture matters more than the formula: `A x*` is the closest reachable point to `b`.

In [ ]:
# === 6.1 Least squares and Moore-Penrose pseudo-inverse ===
A = np.array([[1.0, 2.0],
              [3.0, 4.0],
              [5.0, 6.0]])
b = np.array([1.0, 2.0, 2.5])

x_star = np.linalg.pinv(A) @ b
projection = A @ x_star
residual = b - projection

print("least-squares x* =", x_star)
print("A x* =", projection)
print("residual =", residual)
print("residual norm =", np.linalg.norm(residual))

# Residual should be orthogonal to col(A)
print("A^T residual =", A.T @ residual)

# Projection matrix onto col(A)
P_col = A @ np.linalg.pinv(A)
print("\nProjection matrix P = A A^+ =\n", P_col)
print("P^2 == P ?", np.allclose(P_col @ P_col, P_col))
print("P^T == P ?", np.allclose(P_col.T, P_col))
print("P b =", P_col @ b)
print("matches A x* ?", np.allclose(P_col @ b, projection))

---
## 7. LU, Determinant, and Elimination

LU is Gaussian elimination with bookkeeping. It is the decomposition that turns a general square system into two triangular systems.

Learning move: factor once, solve many times.

In [ ]:
# === 7.1 LU without pivoting for a friendly example ===
A = np.array([[2.0, 1.0, 1.0],
              [4.0, 3.0, 3.0],
              [8.0, 7.0, 9.0]])

L, U = lu_no_pivot(A)
print("A =\n", A)
print("L =\n", L)
print("U =\n", U)
print("L @ U == A ?", np.allclose(L @ U, A))

det_A = np.linalg.det(A)
det_from_U = np.prod(np.diag(U))
print("det(A) from numpy =", det_A)
print("det from U diagonal product =", det_from_U)

# Solve using the factorization idea
b = np.array([1.0, 1.0, 1.0])
y = np.linalg.solve(L, b)
x = np.linalg.solve(U, y)
print("solution x from LU solve =", x)
print("A x =", A @ x)

---
## 8. QR and Eigendecomposition

QR is the orthogonalization decomposition. Eigendecomposition is the invariant-direction decomposition. Together, they give two of the main structural views of a matrix.

Use QR when orthogonality and least squares matter. Use eigendecomposition when invariant directions and repeated action matter.

In [ ]:
# === 8.1 QR least squares and eigendecomposition ===
A = np.array([[1.0, 1.0],
              [1.0, 2.0],
              [1.0, 3.0]])
b = np.array([1.0, 2.0, 2.0])
Q, R = np.linalg.qr(A, mode='reduced')
x_qr = np.linalg.solve(R, Q.T @ b)

print("Q =\n", Q)
print("R =\n", R)
print("Q^T Q =\n", Q.T @ Q)
print("least-squares solution via QR =", x_qr)
print("A x_qr =", A @ x_qr)

S = np.array([[2.0, 1.0],
              [1.0, 2.0]])
eigvals, eigvecs = np.linalg.eigh(S)
print("\nSymmetric matrix S =\n", S)
print("eigenvalues =", eigvals)
print("eigenvectors =\n", eigvecs)
print("S reconstructed =\n", eigvecs @ np.diag(eigvals) @ eigvecs.T)

# Matrix power through eigendecomposition
S4 = eigvecs @ np.diag(eigvals**4) @ eigvecs.T
print("S^4 via eigendecomposition =\n", S4)

dominant_eval, dominant_vec = power_iteration(S)
print("dominant eigenvalue via power iteration ~", dominant_eval)
print("dominant eigenvector ~", dominant_vec)

---
## 9. SVD, Low Rank, and LoRA Intuition

SVD is the decomposition that always exists. It tells you how a matrix rotates, scales, and rotates again. It is also the cleanest way to think about low-rank approximation.

This is where the connection to LoRA becomes operational: if most useful change lives in a few singular directions, a low-rank update can capture it.

In [ ]:
# === 9.1 SVD and best rank-1 approximation ===
A = np.array([[3.0, 2.0, 2.0],
              [2.0, 3.0, -2.0]])
U, S, Vt = np.linalg.svd(A, full_matrices=False)
A1, singular_values = truncated_svd(A, rank=1)

print("singular values =", singular_values)
print("best rank-1 approximation A1 =\n", A1)
print("frobenius error ||A - A1||_F =", np.linalg.norm(A - A1, ord='fro'))
print("sigma_2 =", singular_values[1])

# LoRA-style rank-r update
d = 6
r = 2
B = np.random.randn(d, r)
A_factor = np.random.randn(r, d)
delta_W = B @ A_factor
svals_delta = np.linalg.svd(delta_W, compute_uv=False)
print("\nLoRA-style update shape:", delta_W.shape)
print("rank(delta_W) <= r ?", np.linalg.matrix_rank(delta_W) <= r)
print("singular values of delta_W =", svals_delta)

if HAS_MPL:
    plt.figure(figsize=(5, 3))
    plt.plot(np.arange(1, len(singular_values) + 1), singular_values, marker='o')
    plt.title('Singular Values of A')
    plt.xlabel('Index')
    plt.ylabel('sigma_i')
    plt.grid(True, alpha=0.3)
    plt.show()

---
## 10. Cholesky and Symmetric Positive Definite Matrices

If a matrix is symmetric positive definite, Cholesky is usually the first decomposition to try. It is faster than LU, numerically stable, and closely tied to covariance matrices and Gaussian models.

This is one of the strongest examples of structure paying for itself computationally.

In [ ]:
# === 10.1 Cholesky factorization and SPD solves ===
A = np.array([[4.0, 2.0, 2.0],
              [2.0, 5.0, 1.0],
              [2.0, 1.0, 3.0]])
b = np.array([1.0, 0.0, 1.0])
L = np.linalg.cholesky(A)

print("A =\n", A)
print("L =\n", L)
print("L @ L.T =\n", L @ L.T)

y = np.linalg.solve(L, b)
x = np.linalg.solve(L.T, y)
print("solution x =", x)
print("A x =", A @ x)

logdet = 2 * np.sum(np.log(np.diag(L)))
print("log det(A) from Cholesky =", logdet)
print("log det(A) from slogdet =", np.linalg.slogdet(A)[1])

z = np.random.randn(3)
sample = L @ z
print("\nstandard Gaussian z =", z)
print("correlated sample Lz =", sample)

---
## 11. GEMM vs GEMV and the Hardware View

The same mathematics can produce very different hardware behavior. Large batched matrix-matrix multiply is high reuse and can keep tensor units busy. Matrix-vector multiply has much less reuse and is often bandwidth-bound.

This is one reason training throughput and single-token decoding latency behave so differently.

In [ ]:
# === 11.1 Arithmetic intensity intuition for GEMV vs GEMM ===
def gemv_intensity(m, n):
    flops = 2 * m * n
    bytes_moved = 8 * (m * n + n + m)  # rough float64 accounting for intuition only
    return flops / bytes_moved


def gemm_intensity(m, p, n):
    flops = 2 * m * p * n
    bytes_moved = 8 * (m * p + p * n + m * n)
    return flops / bytes_moved


m, n, p = 4096, 4096, 4096
print("Approx arithmetic intensity GEMV(4096, 4096) =", gemv_intensity(m, n))
print("Approx arithmetic intensity GEMM(4096, 4096, 4096) =", gemm_intensity(m, p, n))

# Batching converts many GEMV-like workloads toward GEMM.
for batch in [1, 4, 16, 64]:
    intensity = gemm_intensity(batch, 4096, 4096)
    print(f"batch={batch:2d} -> batched matmul intensity ~ {intensity:.4f}")

print("\nInterpretation:")
print("Higher arithmetic intensity usually means better chance of being compute-bound.")
print("Lower arithmetic intensity usually means memory traffic dominates.")

---
## 12. What to Notice

The main conceptual moves in this notebook are:

- matrix multiplication is composition, not just a formula
- transpose and inverse reverse order because composition is order-sensitive
- pseudo-inverse is really about projection and least squares
- QR exposes orthogonality, eigendecomposition exposes invariant directions, SVD exposes full low-rank structure
- Cholesky is what happens when symmetry and positive definiteness make the problem easier
- the AI systems view is not separate from the math: GEMM, GEMV, low-rank updates, and numerical conditioning are the math made operational

The next clean step after this notebook is to go deeper on spectral structure: eigenvalues, eigenvectors, singular values, and what they say about dynamics, compression, and optimization.